# Week 1: PyTorch Basics
Following the 30 minute 'Learn the Basics' tutorial.

In [ ]:
import torch
x = torch.rand(3, 3)
y = torch.rand(3, 3)
z = x + y
z.backward(torch.ones_like(z))
print('z', z)
print('x.grad', x.grad)

## 2-layer MLP on MNIST

In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_set = datasets.MNIST('data', train=True, download=True,
                               transform=transforms.ToTensor())
test_set = datasets.MNIST('data', train=False, download=True,
                              transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=1000)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = MLP().to(device)
opt = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        opt.zero_grad()
        out = model(data)
        loss = loss_fn(out, target)
        loss.backward()
        opt.step()

model.eval()
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        pred = model(data).argmax(1)
        correct += (pred == target).sum().item()

print('Accuracy:', correct/len(test_set))

## Visualize Computational Graph

In [ ]:
from torchviz import make_dot
sample = torch.randn(1, 1, 28, 28, device=device)
y = model(sample)
make_dot(y, params=dict(model.named_parameters()))